# Notebook 05 - Incremental Load

## Objective

Load only new records from CSV files into the Silver Delta Tables.

This notebook:
- Reads latest CSV files
- Compares with existing Silver tables
- Inserts only new records

In [0]:
from pyspark.sql.functions import *

In [0]:
volume_path = "/Volumes/dbacademy/default/raw_data"

incremental_tables = [
    {
        "file_name": "biotech_funding.csv",
        "silver_table": "biotech_funding_silver",
        "primary_key": "deal_id"
    },
    {
        "file_name": "clinical_trials.csv",
        "silver_table": "clinical_trials_silver",
        "primary_key": "trial_id"
    },
    {
        "file_name": "disease_burden.csv",
        "silver_table": "disease_burden_silver",
        "primary_key": "year"
    },
    {
        "file_name": "drug_approvals.csv",
        "silver_table": "drug_approvals_silver",
        "primary_key": "approval_id"
    },
    {
        "file_name": "pharma_companies_financials.csv",
        "silver_table": "pharma_companies_silver",
        "primary_key": "company_name"
    }
]

In [0]:
for table in incremental_tables:

    print("=" * 80)
    print(f"Processing {table['file_name']}")

    # Read latest CSV
    source_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{volume_path}/{table['file_name']}")
    )

    # Read Silver table
    target_df = spark.table(
        f"workspace.default.{table['silver_table']}"
    )

    # Find only new records
    incremental_df = source_df.join(
        target_df,
        on=table["primary_key"],
        how="left_anti"
    )

    print("New Records :", incremental_df.count())

    # Append only new records
    incremental_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            f"workspace.default.{table['silver_table']}"
        )

    print("Incremental Load Completed")

In [0]:
for table in incremental_tables:

    count = spark.table(
        f"workspace.default.{table['silver_table']}"
    ).count()

    print(f"{table['silver_table']} : {count}")

In [0]:
%sql

SELECT COUNT(*) FROM workspace.default.biotech_funding_silver;

SELECT COUNT(*) FROM workspace.default.clinical_trials_silver;

SELECT COUNT(*) FROM workspace.default.disease_burden_silver;

SELECT COUNT(*) FROM workspace.default.drug_approvals_silver;

SELECT COUNT(*) FROM workspace.default.pharma_companies_silver;